# Week 10, Day 5: Two-dimensional convolution: receptive fields and convolutional networks

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w010/d5.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Two-dimensional cross-correlation

Write `corr2d(img, k)` for valid mode with two loops over output positions, each computing the sum of the kernel times the patch beneath it. Test it with a small edge detector on an image of a bright square.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def corr2d(img, k):
    H, W = img.shape
    h, w = k.shape
    out = np.zeros((H - h + 1, W - w + 1))
    for i in range(out.shape[0]):
        for j in range(out.shape[1]):
            out[i, j] = ...
    return out

square = np.zeros((20, 20)); square[6:14, 6:14] = 1.0
edge = np.array([[-1.0, 0.0, 1.0]] * 3)           # responds to left-to-right brightness increases
fig, ax = plt.subplots(1, 2, figsize=(6, 3))
ax[0].imshow(square, cmap="gray"); ax[0].set_title("image")
ax[1].imshow(corr2d(square, edge), cmap="RdBu_r"); ax[1].set_title("vertical-edge response")
plt.tight_layout(); plt.show()

In [ ]:
import numpy as np
def test_shape():
    "valid output has size (H - h + 1, W - w + 1)"
    assert corr2d(np.zeros((10, 12)), np.zeros((3, 5))).shape == (8, 8)

def test_values():
    "each output is the sum of kernel times patch"
    r = np.random.default_rng(0)
    X, K = r.normal(size=(7, 6)), r.normal(size=(3, 2))
    Y = corr2d(X, K)
    assert np.isclose(Y[2, 3], np.sum(K * X[2:5, 3:5]))

def test_identity():
    "a kernel with a single 1 at its centre copies the image (shrunk by the border)"
    X = np.arange(25.0).reshape(5, 5)
    K = np.zeros((3, 3)); K[1, 1] = 1
    assert np.allclose(corr2d(X, K), X[1:4, 1:4])

test_shape()
test_values()
test_identity()
print("All checks passed")

### Exercise: An orientation-tuned filter bank

Using the Gabor and bar functions provided, compute for each bar orientation in `oris` the response energy (sum of squared outputs of `corr2d`) of each filter orientation, storing a 4 by 4 list in `energy`. Store the best-matching filter orientation for each bar in `best`, and plot one row as an orientation tuning curve.

In [ ]:
def gabor(theta_deg, size=15, sigma=3.0, wavelength=6.0):
    th = np.radians(theta_deg); r = size // 2
    y, x = np.mgrid[-r:r + 1, -r:r + 1]; y = -y
    across = -x * np.sin(th) + y * np.cos(th)          # distance across the preferred orientation
    along = x * np.cos(th) + y * np.sin(th)
    g = np.exp(-(across ** 2 + along ** 2) / (2 * sigma ** 2)) * np.cos(2 * np.pi * across / wavelength)
    return g - g.mean()

def bar(theta_deg, N=40, width=1.5):
    th = np.radians(theta_deg); y, x = np.mgrid[0:N, 0:N]; x = x - N / 2; y = -(y - N / 2)
    across = -x * np.sin(th) + y * np.cos(th); along = x * np.cos(th) + y * np.sin(th)
    return ((np.abs(across) < width) & (np.abs(along) < 14)).astype(float)

oris = [0, 45, 90, 135]
energy = ...
best = ...

fig, ax = plt.subplots(1, 5, figsize=(10, 2.2))
for a, o in zip(ax[:4], oris):
    a.imshow(gabor(o), cmap="RdBu_r"); a.set_title(f"{o}°"); a.axis("off")
ax[4].plot(oris, energy[1], "o-"); ax[4].set_title("response to a 45° bar"); ax[4].set_xticks(oris)
plt.tight_layout(); plt.show()
print("best filter for each bar:", best)

In [ ]:
import numpy as np
def test_kernel():
    "gabor returns a zero-mean 15 by 15 kernel"
    g = gabor(30)
    assert g.shape == (15, 15) and abs(g.mean()) < 1e-12

def test_energy():
    "energy[b][o] is the summed squared response of filter o to bar b"
    assert np.isclose(energy[1][2], np.sum(corr2d(bar(oris[1]), gabor(oris[2])) ** 2))

def test_choice():
    "every bar is matched best by the filter of its own orientation"
    assert best == oris

test_kernel()
test_energy()
test_choice()
print("All checks passed")

### Exercise: Equivariance and pooling

Shift a 45° bar image three pixels to the right with `np.roll` and convolve both with the 45° Gabor, giving `R_orig` and `R_shift`. Compare a single unit at position (12, 12) before and after, and the maximum over the window of rows and columns 8 to 15 (`pooled_orig`, `pooled_shift`).

In [ ]:
img = bar(45)
shifted = np.roll(img, 3, axis=1)
R_orig = ...
R_shift = ...
pooled_orig = ...
pooled_shift = ...
print("single unit:", R_orig[12, 12].round(2), "->", R_shift[12, 12].round(2), "   pooled:", pooled_orig.round(2), "->", pooled_shift.round(2))

In [ ]:
import numpy as np
def test_equivariant():
    "the response map of the shifted image is the original map shifted by 3 columns"
    assert np.allclose(R_orig[:, :20], R_shift[:, 3:23])

def test_single_unit():
    "a single unit at a fixed position changes by more than 5 when the bar moves"
    assert abs(R_orig[12, 12] - R_shift[12, 12]) > 5

def test_pooled():
    "the max-pooled response over the central 8 by 8 window is unchanged by the shift"
    assert np.isclose(pooled_orig, pooled_shift, rtol=0.05)

test_equivariant()
test_single_unit()
test_pooled()
print("All checks passed")